In [1]:
!pip install -q transformers datasets accelerate evaluate scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.7 MB/s eta 0:00:00


In [2]:
import numpy as np
import torch

from datasets import load_from_disk

from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments
)

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support
)

In [4]:
!ls /content/drive

MyDrive


In [5]:
!ls /content/drive/MyDrive

 24BCS054_HF.pdf
 24BCS054messfee.pdf
'BHANUPANTHULA GANESH KARTHIK10th.pdf'
'BHANUPANTHULA GANESH KARTHIK12th.pdf'
'BHANUPANTHULA GANESH KARTHIKalotment.pdf'
'BHANUPANTHULA GANESH KARTHIKantidrug.pdf'
'BHANUPANTHULA GANESH KARTHIKantiragging (1).pdf'
'BHANUPANTHULA GANESH KARTHIKantiragging.pdf'
'BHANUPANTHULA GANESH KARTHIKfeereceipt.pdf'
'BHANUPANTHULA GANESH KARTHIKjeescore.pdf'
'BHANUPANTHULA GANESH KARTHIKmedical.pdf'
'BHANUPANTHULA GANESH KARTHIKpg.pdf'
'BHANUPANTHULA GANESH KARTHIKpic.jpeg'
'BHANUPANTHULA GANESH KARTHIKsign.jpeg'
'BHANUPANTHULA GANESH KARTHIKundertakubngc.pdf'
'Colab Notebooks'
'Consider my hostel  menu and give me the scadule.gdoc'
 gk-may2026.pdf
'GK pics'
'iiit jabalpur offer letter.pdf'
 index.gdoc
'Joint Entrance Examination (Main) _ India (1).pdf'
'LAST PAY CERTIFICATE.gdoc'
 medical.pdf
'Programming Concepts List.gdoc'
'Projection_solid_HW[1].gdoc'
'Untitled document (1).gdoc'
'Untitled document (2).gdoc'
'Untitled document (3).gdoc'
'Untitled document.g

In [6]:
!ls /content/drive/Shareddrives

ls: cannot access '/content/drive/Shareddrives': No such file or directory


In [7]:
stance_dataset = load_from_disk(
    "/content/drive/MyDrive/AI-News-Perspective-Analyzer/tokenized_stance"
)

stance_dataset

Dataset({
    features: ['text', 'label', 'input_ids', 'attention_mask'],
    num_rows: 48437
})

In [8]:
dataset = stance_dataset.train_test_split(
    test_size=0.2,
    seed=42
)

train_dataset = dataset["train"]
eval_dataset = dataset["test"]

In [9]:
MODEL_NAME = "roberta-base"

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=4
)

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  499MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [10]:
def compute_metrics(eval_pred):

    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=-1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="weighted"
    )

    accuracy = accuracy_score(labels, predictions)

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

In [11]:
training_args = TrainingArguments(

    output_dir="./stance_model",

    eval_strategy="epoch",

    save_strategy="epoch",

    learning_rate=2e-5,

    per_device_train_batch_size=8,

    per_device_eval_batch_size=8,

    num_train_epochs=3,

    weight_decay=0.01,

    fp16=True,

    logging_steps=100,

    load_best_model_at_end=True,

    metric_for_best_model="f1",

    report_to="none"
)

In [12]:
trainer = Trainer(

    model=model,

    args=training_args,

    train_dataset=train_dataset,

    eval_dataset=eval_dataset,

    compute_metrics=compute_metrics
)

In [13]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.197354,0.176597,0.951383,0.946594,0.951383,0.947621
2,0.112389,0.092488,0.980078,0.979530,0.980078,0.979211
3,0.037309,0.070737,0.987510,0.987393,0.987510,0.987408


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=14532, training_loss=0.14646017073994327, metrics={'train_runtime': 3464.5925, 'train_samples_per_second': 33.553, 'train_steps_per_second': 4.194, 'total_flos': 3.058642008881971e+16, 'train_loss': 0.14646017073994327, 'epoch': 3.0})

In [1]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("roberta-base")

tokenizer.save_pretrained(
    "/content/drive/MyDrive/AI-News-Perspective-Analyzer/models/stance_model"
)

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

('/content/drive/MyDrive/AI-News-Perspective-Analyzer/models/stance_model/tokenizer_config.json',
 '/content/drive/MyDrive/AI-News-Perspective-Analyzer/models/stance_model/tokenizer.json')

In [2]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    "/content/drive/MyDrive/AI-News-Perspective-Analyzer/models/stance_model"
)

print("Tokenizer loaded successfully!")
print(type(tokenizer))

Tokenizer loaded successfully!
<class 'transformers.models.roberta.tokenization_roberta.RobertaTokenizer'>


In [14]:
trainer.evaluate()

Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1
0.037309,0.070737,3,0.987510,0.987393,0.987510,0.987408


{'eval_loss': 0.07073653489351273,
 'eval_accuracy': 0.9875103220478944,
 'eval_precision': 0.987393036326143,
 'eval_recall': 0.9875103220478944,
 'eval_f1': 0.9874081566671756}

In [15]:
trainer.save_model(
    "/content/drive/MyDrive/AI-News-Perspective-Analyzer/models/stance_model"
)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]